# SUBLIME — Kaggle Reproduction

Paper: [Towards Unsupervised Deep Graph Structure Learning (WWW 2022)](https://arxiv.org/abs/2201.06367)

This notebook reproduces the paper's results on **Cora / Citeseer / Pubmed** using the *exact* dependencies from the original repo:

| Package        | Version |
|----------------|---------|
| Python         | 3.8     |
| PyTorch        | 1.7.1 (+cu110) |
| DGL            | 0.7.1 (+cu110) |
| numpy          | 1.20.2  |
| scipy          | 1.6.3   |
| scikit-learn   | 0.24.2  |
| munkres        | 1.1.4   |

### How to use on Kaggle
1. Settings → **Accelerator: GPU T4 x2** (or **GPU P100**). Do **not** use L4/A100 — PyTorch 1.7.1 cu110 was not compiled for `sm_89`/`sm_90`.
2. Run cells 1–4 once to build the environment (takes ~5 min).
3. Run any of the experiment cells in section 5.

### Expected results (paper Table 2 / Table 3)

| Dataset | Mode | Task | Reported |
|---------|------|------|----------|
| Cora     | Structure Inference  | Classification     | 83.6 ± 0.4 |
| Cora     | Structure Refinement | Classification     | 84.7 ± 0.3 |
| Cora     | Structure Refinement | Clustering (ACC)   | 71.1 |
| Citeseer | Structure Inference  | Classification     | 73.6 ± 0.4 |
| Citeseer | Structure Refinement | Classification     | 73.9 ± 0.5 |
| Citeseer | Structure Refinement | Clustering (ACC)   | 68.7 |
| Pubmed   | Structure Inference  | Classification     | 80.2 ± 0.5 |
| Pubmed   | Structure Refinement | Classification     | 81.0 ± 0.4 |


## 1. Sanity check the Kaggle runtime

In [ ]:
import subprocess, sys
print('Default Python:', sys.version.split()[0])
print()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'],
                    capture_output=True, text=True).stdout)


## 2. Build the paper environment (Python 3.8 + PyTorch 1.7.1 + DGL 0.7.1)

We use **micromamba** only to get a Python 3.8 interpreter (Kaggle ships 3.10/3.12). Everything else is installed from official pip wheels:

* `torch==1.7.1+cu110` from PyTorch's wheel index
* `dgl-cu110==0.7.1` from DGL's wheel index

Both wheels are still hosted and compile-free. Run this cell **once per Kaggle session**.


In [ ]:
import os, subprocess, glob

ENV = '/tmp/sublime-env'
PY  = f'{ENV}/bin/python'
PYX = f'{ENV}/bin/sublime-python'   # wrapper that sets LD_LIBRARY_PATH
MM  = '/tmp/micromamba'

def sh(cmd, check=True):
    print('>>>', cmd if len(cmd) < 120 else cmd[:117] + '...')
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        print(r.stdout[-2500:])
    if check and r.returncode != 0:
        raise RuntimeError(f'failed (rc={r.returncode}): {cmd}')

# --- 1. micromamba (single static binary) ------------------------------------
if not os.path.exists(MM):
    sh('curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest '
       '| tar -xvj --strip-components=1 -C /tmp bin/micromamba')
    sh(f'chmod +x {MM}')

# --- 2. Python 3.8 env (no CUDA — we get CUDA libs from NVIDIA pip wheels) ---
if not os.path.exists(PY):
    sh(f'{MM} create -p {ENV} -y -c conda-forge --no-rc python=3.8 pip')

# --- 3. NVIDIA CUDA 11 runtime libraries via official pip wheels -------------
#  These wheels are published by NVIDIA on PyPI and contain the exact .so files
#  DGL 0.7.1 dlopens (libcublas.so.11, libcudart.so.11.0, libcusparse.so.11,
#  libcurand.so.10, libcusolver.so.10, libcufft.so.10, libcudnn.so.8).
sh(f'{PY} -m pip install --quiet '
   f'nvidia-cuda-runtime-cu11 '
   f'nvidia-cublas-cu11 '
   f'nvidia-cusparse-cu11 '
   f'nvidia-curand-cu11 '
   f'nvidia-cusolver-cu11 '
   f'nvidia-cufft-cu11 '
   f'nvidia-cudnn-cu11')

# --- 4. PyTorch 1.7.1 + CUDA 11.0 (official wheel) ---------------------------
sh(f'{PY} -m pip install --quiet '
   f'torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 '
   f'-f https://download.pytorch.org/whl/torch_stable.html')

# --- 5. DGL 0.7.1 + CUDA 11.0 (official DGL wheel index) ---------------------
sh(f'{PY} -m pip install --quiet '
   f'dgl-cu110==0.7.1 '
   f'-f https://data.dgl.ai/wheels/repo.html')

# --- 6. Exact paper Python deps ----------------------------------------------
sh(f'{PY} -m pip install --quiet '
   f'numpy==1.20.2 scipy==1.6.3 scikit-learn==0.24.2 '
   f'munkres==1.1.4 networkx==2.5 ogb==1.3.1')

# --- 7. Wrapper that sets LD_LIBRARY_PATH so dlopen finds the CUDA libs ------
nvidia_libs = sorted(set(os.path.dirname(p) for p in
    glob.glob(f'{ENV}/lib/python3.8/site-packages/nvidia/*/lib/lib*.so*')))
torch_lib = f'{ENV}/lib/python3.8/site-packages/torch/lib'
ld_paths  = ':'.join(nvidia_libs + [torch_lib])
print('LD_LIBRARY_PATH entries:')
for p in nvidia_libs + [torch_lib]:
    print(' ', p)

wrapper = (
    '#!/bin/sh\n'
    f'export LD_LIBRARY_PATH="{ld_paths}:${{LD_LIBRARY_PATH}}"\n'
    f'exec {PY} "$@"\n'
)
with open(PYX, 'w') as f:
    f.write(wrapper)
os.chmod(PYX, 0o755)
print('wrapper written to', PYX)

# --- 8. Quick sanity: can we find libcublas.so.11? ---------------------------
import subprocess as _sp
print()
print('libcublas.so.11 search:')
print(_sp.run(f'find {ENV}/lib/python3.8/site-packages/nvidia -name "libcublas.so.11"',
              shell=True, text=True, capture_output=True).stdout or '  (not found!)')


## 3. Verify the environment

In [ ]:
# Verification: run inside the paper env via the LD_LIBRARY_PATH wrapper.
verify_src = """
import torch, dgl, numpy, scipy, sklearn, sys
print("Python    :", sys.version.split()[0])
print("PyTorch   :", torch.__version__)
print("CUDA      :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
print("DGL       :", dgl.__version__)
print("numpy     :", numpy.__version__)
print("scipy     :", scipy.__version__)
print("sklearn   :", sklearn.__version__)
"""
with open('/tmp/_sublime_verify.py', 'w') as f:
    f.write(verify_src)

!/tmp/sublime-env/bin/sublime-python /tmp/_sublime_verify.py


## 4. Clone the repo and cd into it

In [ ]:
import os
REPO_DIR = '/kaggle/working/SUBLIME'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/wishaalk/SUBLIME.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
!ls data/ | head


## 5. Reproduce the paper

Each cell below corresponds to one row in Table 2 / Table 3 of the paper and uses **exactly** the
hyperparameters from `scripts/*.sh`. We just swap `python` for our paper-env interpreter.


### 5a. Cora — Classification @ Structure Inference (Table 2)
_(mirrors `scripts/cora_si.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset cora -ntrials 5 -sparse 0 -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 -type_learner fgp -k 30 -sim_function cosine -activation_learner relu -gsl_mode structure_inference -eval_freq 20 -tau 1 -maskfeat_rate_learner 0.5 -maskfeat_rate_anchor 0.7 -contrast_batch_size 0 -c 0 -gpu 0

### 5b. Cora — Classification @ Structure Refinement (Table 2)
_(mirrors `scripts/cora_sr.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset cora -ntrials 5 -sparse 0 -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.75 -nlayers_cls 2 -patience_cls 10 -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 -type_learner fgp -k 30 -sim_function cosine -activation_learner relu -gsl_mode structure_refinement -eval_freq 50 -tau 0.9999 -maskfeat_rate_learner 0.7 -maskfeat_rate_anchor 0.6 -contrast_batch_size 0 -c 0 -gpu 0

### 5c. Cora — Clustering @ Structure Refinement (Table 3)
_(mirrors `scripts/cora_clu.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset cora -downstream_task clustering -ntrials 10 -sparse 0 -epochs 2500 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 -type_learner fgp -k 20 -sim_function cosine -activation_learner relu -gsl_mode structure_refinement -eval_freq 100 -tau 0.9999 -maskfeat_rate_learner 0.1 -maskfeat_rate_anchor 0.8 -contrast_batch_size 0 -c 0 -gpu 0

### 5d. Citeseer — Classification @ Structure Inference (Table 2)
_(mirrors `scripts/citeseer_si.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset citeseer -ntrials 5 -sparse 0 -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 -epochs 1000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 -type_learner att -k 20 -sim_function cosine -activation_learner tanh -gsl_mode structure_inference -eval_freq 50 -tau 0.9999 -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.7 -contrast_batch_size 0 -c 0 -gpu 0

### 5e. Citeseer — Classification @ Structure Refinement (Table 2)
_(mirrors `scripts/citeseer_sr.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset citeseer -ntrials 5 -sparse 0 -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 -type_learner att -k 20 -sim_function cosine -activation_learner tanh -gsl_mode structure_refinement -eval_freq 20 -tau 0.9999 -maskfeat_rate_learner 0.6 -maskfeat_rate_anchor 0.8 -contrast_batch_size 0 -c 0 -gpu 0

### 5f. Citeseer — Clustering @ Structure Refinement (Table 3)
_(mirrors `scripts/citeseer_clu.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset citeseer -downstream_task clustering -ntrials 10 -sparse 0 -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 -type_learner att -k 20 -sim_function cosine -activation_learner tanh -gsl_mode structure_refinement -eval_freq 100 -tau 0.999 -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.9 -contrast_batch_size 0 -c 0 -gpu 0

### 5g. Pubmed — Classification @ Structure Inference (Table 2)
_(mirrors `scripts/pubmed_si.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset pubmed -ntrials 5 -sparse 1 -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 -epochs 2000 -lr 0.01 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 -type_learner att -k 15 -sim_function cosine -activation_learner tanh -gsl_mode structure_inference -eval_freq 20 -tau 1 -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.3 -contrast_batch_size 2000 -c 0 -gpu 0

### 5h. Pubmed — Classification @ Structure Refinement (Table 2)
_(mirrors `scripts/pubmed_sr.sh`)_

In [ ]:
!/tmp/sublime-env/bin/sublime-python main.py -dataset pubmed -ntrials 5 -sparse 1 -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 -epochs 1500 -lr 0.001 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 -type_learner mlp -k 10 -sim_function cosine -activation_learner relu -gsl_mode structure_refinement -eval_freq 20 -tau 0.999 -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.4 -contrast_batch_size 2000 -c 50 -gpu 0

---
### Notes
* Each `ntrials=5` classification run on Cora/Citeseer takes ~10–20 min on a T4. Pubmed `ntrials=5` takes ~30–60 min.
* If a run dies with `CUDA error: no kernel image is available for execution on the device`, the Kaggle GPU is too new (e.g. L4). Switch the accelerator to **T4 x2** or **P100**.
* If a `pip install` line ever fails (DGL retires old wheels), pin the closest available patch — the paper's results were obtained with DGL `0.7.x`.
